In [14]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA, AirQuality
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from einops import rearrange, repeat
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger
from utils import MaskedRMSE
from dataset_utils import SDWPE



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(43)

43

In [15]:
# dataset = MetrLA(root='./data/metrla')

# connectivity = dataset.get_connectivity(threshold=0.1,
#                                         include_self=False,
#                                         # normalize_axis=1,
#                                         force_symmetric=False,
#                                         layout="edge_index")

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=connectivity,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)
# print(torch_dataset)

In [16]:
dataset = AirQuality(root='./data/aq', impute_nans=True, small=False)

splitting = {"val_len": 0.1,
            "test_len": 0.2}


connectivity_sparse= {"method": "distance",
                    "threshold": 0.1,
                    "include_self": False,
                    "layout": "edge_index"}

adj = dataset.get_connectivity(**connectivity_sparse)

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=adj,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)

torch_dataset

SpatioTemporalDataset(n_samples=8737, n_nodes=437, n_channels=1)

In [17]:
# dataset = SDWPE()

# splitting = {"val_len": 0.1,
#             "test_len": 0.2}


# connectivity_sparse= {"method": "distance",
#                     "threshold": 0.1,
#                     "include_self": False,
#                     "layout": "edge_index"}

# adj = dataset.get_connectivity(**connectivity_sparse)

# covariates = {'u': dataset.datetime_encoded('day').values}

# torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
#                                       connectivity=adj,
#                                       mask=dataset.mask,
#                                       covariates=covariates,
#                                       horizon=12,
#                                       window=12,
#                                       stride=1)

# torch_dataset

In [18]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=16,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=6279}
{Validation dataloader: size=687}
{Test dataloader: size=1747}
{Predict dataloader: None}


In [19]:
def conv_with_norm(input_size, hidden_size, kernel_size, stride, groups):
    modules = nn.Sequential(
        nn.Conv1d(in_channels = input_size,
                     out_channels = hidden_size,
                     kernel_size=kernel_size,
                     stride=stride,
                     groups=groups),
        nn.GELU(),
        nn.BatchNorm1d(hidden_size),
    )
    return modules



class DW_large_kernel(nn.Module):
    def __init__(self,
                 input_size,hidden_size,
                 large_kernel,
                 stride, groups):
        super().__init__()

        self.hidden_size = hidden_size
        self.large_kernel = large_kernel
        self.stride = stride


        # only one large kernel
        self.large_conv = conv_with_norm(input_size = input_size, hidden_size = hidden_size,
                                       kernel_size=large_kernel, stride=stride, groups=groups)
    def forward(self, x_emb):
        # causual pad at left side
        x = F.pad(x_emb,
                  pad=((self.large_kernel - 1), 0),
                  mode='constant', value=0)
        out = self.large_conv(x)

        return out

In [20]:
class Backbone_blocks(nn.Module):
    def __init__(self, large_kernel, num_variables, hidden_size, drop=0.1):
        super().__init__()
        self.dw_conv = nn.Sequential(
            DW_large_kernel(num_variables*hidden_size,num_variables*hidden_size,
                              large_kernel,
                              stride=1,
                              groups = num_variables*hidden_size),
            nn.Dropout(p=drop),
            nn.GELU()
 
        )
        
        self.pw_con1 = nn.Sequential(
            nn.Conv1d(
            in_channels=num_variables*hidden_size, 
            out_channels=num_variables*hidden_size, 
            kernel_size=1,
            groups=num_variables
            ),
            nn.Dropout(p=drop),
            nn.GELU()
            
        )
        
        self.pw_con2 = nn.Sequential(
            nn.Conv1d(
            in_channels=num_variables*hidden_size, 
            out_channels=num_variables*hidden_size, 
            kernel_size=1,
            groups=hidden_size
            ),
            nn.Dropout(p=drop),
            nn.GELU()
            
        )

    def forward(self, x_emb):
        # x_emb -> [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce]
        batch_size, num_nodes, feature_dim, D, timesteps_reduce = x_emb.shape
        
        # x_emb torch.Size([32, 10, 207, 5, 32, 12])

        x = rearrange(x_emb, 'b n f d t -> (b n) (f d) t').contiguous() 
        x = self.dw_conv(x)

        x = rearrange(x, '(b n) (f d) t -> (b f) t n d', b=batch_size, n=num_nodes, f=feature_dim, d=D)

        
        x = rearrange(x, '(b f) t n d -> (b n) (f d) t ',b=batch_size, n=num_nodes, f=feature_dim, d=D)

        x = self.pw_con1(x)
        x = rearrange(x, '(b n) (f d) t -> (b n) (d f) t',
                      b=batch_size, n=num_nodes, f=feature_dim, d=D).contiguous() 

        # x = rearrange(x, '(b s) n f d t -> (b s n) (d f) t',
        #               b=batch_size, s=num_samples)

        x = self.pw_con2(x)
        x = rearrange(x, '(b n) (d f) t -> b d n f t',
                      b=batch_size, d=D,
                      n=num_nodes, f=feature_dim).contiguous() 
        
        # [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce]
        x = rearrange(x, 'b d n f t -> b n f d t').contiguous() 
        
        # residual connection
        out = x + x_emb
        
        return out

In [21]:
class ModernTCN(pl.LightningModule):
    def __init__(self,
                 input_size,hidden_size,
                 large_kernel, patch_size,
                 windows, horizon, dropout):
        super().__init__()

        self.patch_size = patch_size

        self.stem = nn.Sequential(
            nn.Conv1d(input_size, hidden_size, kernel_size=patch_size),
            nn.GELU()
        )

        self.num_blocks = len(large_kernel)

        # backbones
        self.blocks = nn.ModuleList()
        for block_id in range(self.num_blocks):
            backbone = Backbone_blocks(large_kernel[block_id],
                                      input_size+2, hidden_size, dropout)
            self.blocks.append(backbone)

        self.head = nn.Sequential(
            nn.Linear((input_size+2)*hidden_size, 128),
            nn.GELU(),
            nn.Linear(128, horizon)
        )

    def single_forward(self, x, u):
        batch_size, timesteps, num_nodes, _, feature_dim = x.shape

        if u.dim() == 3:
            u = repeat(u, 'b t f -> b t n 1 f',n=num_nodes)
        x = torch.cat([x, u], -1)

        *_, feature_dim = x.shape

        for i in range(self.num_blocks):
            # [batch_size, num_samples, timesteps, num_nodes, length, D, feature_dim] 
            # -> [batch_size * num_samples * num_nodes * feature_dim, D, timesteps, length] 
            if i == 0:
                x = rearrange(x, 'b t n d f -> (b n f) d t').contiguous() 

                x = self.stem(x) # --> [batch_size * num_nodes * feature_dim, D, timesteps_reduced]
            
                x = rearrange(x, '(b n f) d t -> b n f d t', b=batch_size, n=num_nodes, f=feature_dim).contiguous() 
            
            x = self.blocks[i](x)

        # x = rearrange(x, '(b f) t n d -> b t n d f', b=batch_size, n=num_nodes, f=feature_dim).contiguous()
        return x

    def forward(self, x, u):
        # x --> [batch_size, time_steps, num_nodes, features]
        # u --> [batch_size, time_steps, features]

        x = x.unsqueeze(-2)
        x = self.single_forward(x, u)

        x = x[..., -1]
        x = rearrange(x, 'b n f d -> b n (f d)').contiguous() 
        
        pred = self.head(x)

        pred = rearrange(pred, 'b n h -> b h n').contiguous() 

        return pred.unsqueeze(-1)

In [22]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
        'mae': torch_metrics.MaskedMAE(),
        'rmse': MaskedRMSE()
    }

model = ModernTCN(input_size=1,hidden_size=32, large_kernel = [5, 3, 3], patch_size=1,
                          windows = 12, horizon=12, dropout = 0.1)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [23]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-4
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)

In [24]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=5,
        mode='min',
    min_delta = 0.001
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        # precision = '32',
        check_val_every_n_epoch = 3,
        logger=False

    
)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [25]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | ModernTCN        | 26.6 K | train
-----------------------------------------------------------
26.6 K    Trainable params
0         Non-trainable params
26.6 K    Total params
0.106     Total estimated model params size (MB)
70        Modules in train mode
0         Modules in eval mode


Training: |                                                                                                   …

Arguments ['edge_weight', 'edge_index'] are filtered out. Only args ['u', 'x'] are forwarded to the model (ModernTCN).


Validation: |                                                                                                 …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

In [26]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=29-step=4500-v8.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/checkpoints/epoch=29-step=4500-v8.ckpt


Testing: |                                                                                                    …

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     22.11845588684082     │
│         test_mae          │    22.322694778442383     │
│         test_rmse         │     38.53504180908203     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 22.322694778442383,
  'test_rmse': 38.53504180908203,
  'test_loss': 22.11845588684082}]